In [ ]:
!pip install -q torch==2.4.1 triton==3.0.0

In [ ]:
import torch
import triton
import triton.language as tl

## Preprocessing

In [ ]:
def pack_blocks(mat: torch.Tensor, block_size: int):
    flat = mat.flatten()
    vals = []
    idx_in_block = []
    block_ptr = [0]  # prefix sum over blocks

    for start in range(0, flat.numel(), block_size):
        block = flat[start:start + block_size]
        nz_idx = (block != 0).nonzero(as_tuple=False).squeeze(-1)
        vals.append(block[nz_idx])
        idx_in_block.append(nz_idx)
        block_ptr.append(block_ptr[-1] + nz_idx.numel())

    vals = torch.cat(vals, dim=0) if vals else torch.tensor([], dtype=mat.dtype)
    idx_in_block = torch.cat(idx_in_block, dim=0) if idx_in_block else torch.tensor([], dtype=torch.long)
    block_ptr = torch.tensor(block_ptr, dtype=torch.long)  # len = n_blocks + 1

    return vals, idx_in_block, block_ptr

# Usage
# vals: nonzeros in block order
# idx_in_block: positions 0..block_size-1 within each block
# block_ptr[b]: start offset of block b in vals/idx_in_block
vals, idx_in_block, block_ptr = pack_blocks(A, block_size=4)